# OHAUS SPX222 scale — connect & weigh test

Talks to the SPX222 over its Ethernet (serial-to-TCP) adapter at
`192.168.254.132:9761`. The balance speaks **MT-SICS** — weights come back as
`S S  <weight> <unit>` (`S`=stable, `D`=unstable).

Key rule: if a weigh returns **nothing**, that's a dropped link — the wrapper
returns status `disconnected` with `weight=None`, never a fake `0`.

## Setup

In [2]:
from spx222_driver import SPX222

scale = SPX222(ip="10.0.0.50", port=9761, timeout=3.0)

## Connect

Returns `True` only if the TCP socket opened. `check_connection()` goes further
— it asks the balance for its identity (`I2`) and is `True` only if it actually
answered.

In [3]:
print("connect()        :", scale.connect())
print("check_connection :", scale.check_connection())
print("info             :", scale.info())

connect()        : True
check_connection : True
info             : SPX222 220.90 g


## Weigh (immediate)

`weigh()` sends MT-SICS `SI` and returns a `Reading`. Try it with the pan empty
and with something on it.

In [1]:
import time

for _ in range(1):
    r = scale.weigh()
    print(f"{str(r):24s}  status={r.status:12s} weight={r.weight} unit={r.unit!r}")
    time.sleep(0.5)

NameError: name 'scale' is not defined

In [7]:
r

Reading(status='disconnected', weight=None, unit='', raw='')

## Weigh (wait for stable)

`weigh_stable()` sends `S`, which blocks until the balance settles.

In [4]:
r = scale.weigh_stable(timeout=10.0)
print(r)

200.03 g (stable)


## Disconnect handling

Always branch on `status` (or `connected`), never on the number. A silent
balance reports `disconnected` — unplug the Ethernet adapter and re-run to see
it flip.

In [10]:
r = scale.weigh()

if not r.connected:
    print("scale is DISCONNECTED — do not trust a weight")
elif not r.stable:
    print(f"reading not settled yet: {r}")
else:
    print(f"good stable weight: {r.weight} {r.unit}")

good stable weight: 200.02 g


## Close

In [5]:
scale.close()
print("closed:", not scale.is_connected())

closed: True
